# graph-analysis walkthrough

A short tour of the **pure-numpy reference core** (`bdgraph`): no Spark,
networkx or pyspark needed. We run the one-command demo on a seeded stochastic
block model (SBM) with three planted communities, then look at the structural
diagnostics added to the package: **betweenness centrality**, **k-core
decomposition**, **modularity**, and **degree statistics**.

Everything here is deterministic in the seed and verifiable by eye.

## 1. Run the one-command demo

`run_demo(0)` synthesises the SBM, runs PageRank, connected components, label
propagation and triangle counting, and now also reports the new structural
metrics. The numbers are pinned in `tests/test_demo.py`.

In [ ]:
import numpy as np

from bdgraph.demo import _planted_labels, run_demo, synthesize_sbm

summary = run_demo(0, out_dir="../outputs")
for key in (
    "num_nodes", "num_edges", "num_components",
    "num_communities_found", "num_planted_communities",
    "global_triangles", "avg_clustering",
    "max_core_number", "modularity_found", "mean_degree", "max_degree",
):
    print(f"{key:>26}: {summary[key]}")
print("top PageRank node:", summary["top_pagerank"][0])

## 2. Rebuild the SBM graph

We reconstruct the exact graph the demo used so we can run the individual
algorithms on it.

In [ ]:
sizes = (12, 10, 8)
A = synthesize_sbm(sizes=sizes, seed=0)
planted = _planted_labels(sizes)
print("adjacency shape:", A.shape, " symmetric:", bool((A == A.T).all()))
print("planted labels :", planted.tolist())

## 3. Betweenness centrality (Brandes)

Exact shortest-path betweenness. On this SBM the highest-betweenness node is the
hub that bridges the densest block; it typically coincides with the top PageRank
node.

In [ ]:
from bdgraph import betweenness_centrality

bc = betweenness_centrality(A, normalized=True)
top = np.argsort(bc)[::-1][:5]
for node in top:
    print(f"node {node:2d}: betweenness {bc[node]:.4f}")

## 4. k-core decomposition

The core number of a node is the largest `k` for which it survives in the
`k`-core (every node having degree >= k). Dense-block members sit in the deepest
core; the sparse inter-block links sit shallower.

In [ ]:
from bdgraph import degree_stats, k_core_decomposition

core = k_core_decomposition(A)
print("max core number:", int(core.max()))
print("core numbers   :", core.tolist())

stats = degree_stats(A)
print("mean degree:", round(stats["mean_degree"], 3),
      " max degree:", stats["max_degree"])
print("degree histogram:", stats["histogram"])

## 5. Modularity of the partition

Modularity scores a partition against the configuration-model null. The planted
3-block partition scores high and positive; collapsing everything into one
community scores ~0.

In [ ]:
from bdgraph import label_propagation, modularity

found = label_propagation(A, max_iter=20, seed=0)
print("Q (planted partition) :", round(modularity(A, planted), 4))
print("Q (detected partition):", round(modularity(A, found), 4))
print("Q (single community)  :", round(modularity(A, np.zeros(A.shape[0], int)), 4))

## Takeaways

- Label propagation recovers exactly the 3 planted communities, and the planted
  partition has high positive modularity (~0.56) while a single community gives
  ~0.
- The top-betweenness and top-PageRank nodes agree: both flag the best-connected
  hub of the densest block.
- These are structural descriptions, not explanations. Centrality is conditional
  on the edge set; modularity has a resolution limit. See `USAGE.md` for the
  honest-interpretation notes.